<a href="https://colab.research.google.com/github/LINWOO0099/Pandas_DataFrame/blob/main/pandasdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# task 1
import pandas as pd
import numpy as np

# 1.
students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None, 'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}
enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-02-10', '2024-02-15', '2024-03-01']
}
scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}

df_students = pd.DataFrame(students_data)
df_enrollments = pd.DataFrame(enrollments_data)
df_scores = pd.DataFrame(scores_data)

print("=== TASK 1: Data Preparation ===")

# 2.
null_counts = df_students.isnull().sum()
null_pct = (df_students.isnull().sum() / len(df_students)) * 100
for col in df_students.columns:
    print(f"Column: {col}, Nulls: {null_counts[col]} ({null_pct[col]:.2f}%)")

# 3.
df_students['city'] = df_students['city'].fillna('Unknown')

# 4.
df_students = df_students.dropna(subset=['name'])

print("\nCleaned Students DataFrame:")
print(df_students)

=== TASK 1: Data Preparation ===
Column: student_id, Nulls: 0 (0.00%)
Column: name, Nulls: 1 (14.29%)
Column: email, Nulls: 1 (14.29%)
Column: city, Nulls: 1 (14.29%)

Cleaned Students DataFrame:
   student_id   name            email     city
0         101  Alice  alice@email.com   Mumbai
1         102    Bob    bob@email.com    Delhi
3         104  David             None   Mumbai
4         105   Emma   emma@email.com  Unknown
5         106  Frank  frank@email.com  Chennai
6         107  Grace  grace@email.com    Delhi


In [3]:
# Task 2

# Inner join
inner_merge = pd.merge(df_students, df_enrollments, on='student_id', how='inner')
excluded = df_students[~df_students['student_id'].isin(df_enrollments['student_id'])]['student_id'].tolist()
print(f"Inner Join: {len(inner_merge)} rows.")
print(f"Excluded students: {excluded} - Not in enrollments table.")

# Left Join
left_merge = pd.merge(df_students, df_enrollments, on='student_id', how='left')
null_courses = left_merge[left_merge['course_name'].isna()]['student_id'].tolist()
print(f"\nLeft Join: {len(left_merge)} rows.")
print(f"Students with null course_name: {null_courses} (They haven't enrolled in courses).")

# Right Join
right_merge = pd.merge(df_students, df_enrollments, on='student_id', how='right')
no_name_ids = right_merge[right_merge['name'].isna()]['student_id'].tolist()
print(f"\nRight Join: {len(right_merge)} rows.")
print(f"Student IDs without names: {no_name_ids} (IDs in enrollments but not in cleaned students).")

# Full Outer Join
outer_merge = pd.merge(df_students, df_enrollments, on='student_id', how='outer', indicator=True)
print(f"\nOuter Join: {len(outer_merge)} rows.")
print("\nRows with missing data in name OR course:")
print(outer_merge[outer_merge['name'].isna() | outer_merge['course_name'].isna()])
print("\nMerge Distribution:\n", outer_merge['_merge'].value_counts())

Inner Join: 3 rows.
Excluded students: [104, 106, 107] - Not in enrollments table.

Left Join: 6 rows.
Students with null course_name: [104, 106, 107] (They haven't enrolled in courses).

Right Join: 6 rows.
Student IDs without names: [103, 108, 109] (IDs in enrollments but not in cleaned students).

Outer Join: 9 rows.

Rows with missing data in name OR course:
   student_id   name            email     city course_name enrollment_date  \
2         103    NaN              NaN      NaN      Python      2024-02-01   
3         104  David             None   Mumbai         NaN             NaN   
5         106  Frank  frank@email.com  Chennai         NaN             NaN   
6         107  Grace  grace@email.com    Delhi         NaN             NaN   
7         108    NaN              NaN      NaN          AI      2024-02-15   
8         109    NaN              NaN      NaN      Python      2024-03-01   

       _merge  
2  right_only  
3   left_only  
5   left_only  
6   left_only  
7  right

In [4]:
# task 3


# 1
score_map = df_scores.set_index('student_id')['exam_score'].to_dict()
df_students['exam_score'] = df_students['student_id'].map(score_map)

print("Students with Scores (using map):")
print(df_students[['student_id', 'name', 'exam_score']])



# 2
def auto_merge(df1, df2, join_type, key_column):
    merged = pd.merge(df1, df2, on=key_column, how=join_type)
    return {
        'result_df': merged,
        'row_count': len(merged),
        'join_type': join_type
    }

# 3
test_inner = auto_merge(df_students, df_enrollments, 'inner', 'student_id')
test_left = auto_merge(df_students, df_enrollments, 'left', 'student_id')

print(f"\nAutomation Test (Inner): {test_inner['row_count']} rows")
print(f"Automation Test (Left): {test_left['row_count']} rows")

Students with Scores (using map):
   student_id   name  exam_score
0         101  Alice        85.0
1         102    Bob        92.0
3         104  David        78.0
4         105   Emma        88.0
5         106  Frank        95.0
6         107  Grace         NaN

Automation Test (Inner): 3 rows
Automation Test (Left): 6 rows
